Set C: Output Consistency Evaluation

Research Question:
RQ4: Does quantization affect output consistency across multiple runs with the same prompt?

Evaluation Coverage:
- Consistency metrics: Self-BLEU, lexical diversity, semantic stability
- Answer variation: Token-level and semantic differences across runs
- Category-specific consistency patterns
- Quantization impact on determinism

Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set
import re
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Imports complete")

In [ ]:
INPUT_DIR = Path('/kaggle/input/generation-sets')
OUTPUT_DIR = Path('/kaggle/working/set_c_evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05
CONSISTENCY_RUNS = 5

np.random.seed(RANDOM_SEED)

print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Normalization and Basic Utilities

In [ ]:
def normalize_answer(text: str) -> str:
    """Normalize text following SQuAD evaluation protocol"""
    import unicodedata
    
    if not text:
        return ""
    
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    return text.strip()

def extract_conservative(text: str, ground_truth: str = None, max_ratio: float = 3.0) -> str:
    """Conservative extraction strategy"""
    if not text:
        return ""
    
    text = text.strip()
    
    prefixes = ['Answer:', 'answer:', 'A:', 'a:', 'The answer is:', 'the answer is:']
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
            break
    
    text = text.split('\n')[0].strip()
    
    if ground_truth and text:
        gt_words = len(ground_truth.split())
        text_words = len(text.split())
        
        if gt_words > 0 and text_words > max_ratio * gt_words:
            sentences = re.split(r'[.!?]+', text)
            if sentences and sentences[0].strip():
                text = sentences[0].strip()
    
    return text.strip()

print("Normalization and extraction defined")

Consistency Metrics

In [ ]:
def compute_self_bleu(predictions: List[str]) -> float:
    """
    Compute average BLEU score when each prediction is compared against others.
    Lower Self-BLEU indicates more diversity (less consistency).
    Higher Self-BLEU indicates more consistency.
    """
    if len(predictions) < 2:
        return 1.0
    
    def get_ngrams(tokens: List[str], n: int) -> Counter:
        if len(tokens) < n:
            return Counter()
        return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)])
    
    def bleu_score(reference: List[str], candidate: List[str], max_n: int = 4) -> float:
        scores = []
        
        for n in range(1, max_n + 1):
            ref_ngrams = get_ngrams(reference, n)
            cand_ngrams = get_ngrams(candidate, n)
            
            if not cand_ngrams:
                scores.append(0.0)
                continue
            
            overlap = sum(min(cand_ngrams[ng], ref_ngrams[ng]) for ng in cand_ngrams)
            precision = overlap / sum(cand_ngrams.values())
            scores.append(precision)
        
        if not scores or all(s == 0 for s in scores):
            return 0.0
        
        geometric_mean = np.exp(np.mean([np.log(s) if s > 0 else -np.inf for s in scores]))
        
        ref_len = len(reference)
        cand_len = len(candidate)
        
        if cand_len >= ref_len:
            bp = 1.0
        else:
            bp = np.exp(1 - ref_len / cand_len) if cand_len > 0 else 0.0
        
        return bp * geometric_mean
    
    tokenized = [normalize_answer(p).split() for p in predictions]
    
    scores = []
    for i, candidate in enumerate(tokenized):
        references = [ref for j, ref in enumerate(tokenized) if j != i]
        
        if not references:
            continue
        
        ref_scores = [bleu_score(ref, candidate) for ref in references]
        scores.append(np.mean(ref_scores))
    
    return float(np.mean(scores)) if scores else 0.0

def compute_exact_match_consistency(predictions: List[str]) -> float:
    """
    Measure how often predictions are exactly the same (after normalization).
    Returns fraction of prediction pairs that match exactly.
    """
    if len(predictions) < 2:
        return 1.0
    
    normalized = [normalize_answer(p) for p in predictions]
    
    matches = 0
    comparisons = 0
    
    for i in range(len(normalized)):
        for j in range(i + 1, len(normalized)):
            comparisons += 1
            if normalized[i] == normalized[j]:
                matches += 1
    
    return float(matches / comparisons) if comparisons > 0 else 0.0

def compute_token_overlap_consistency(predictions: List[str]) -> float:
    """
    Measure average Jaccard similarity across all prediction pairs.
    High overlap = high consistency.
    """
    if len(predictions) < 2:
        return 1.0
    
    tokenized = [set(normalize_answer(p).split()) for p in predictions]
    
    overlaps = []
    
    for i in range(len(tokenized)):
        for j in range(i + 1, len(tokenized)):
            intersection = len(tokenized[i] & tokenized[j])
            union = len(tokenized[i] | tokenized[j])
            
            if union > 0:
                overlaps.append(intersection / union)
            else:
                overlaps.append(1.0)
    
    return float(np.mean(overlaps)) if overlaps else 0.0

def compute_length_variance(predictions: List[str]) -> float:
    """
    Coefficient of variation in answer length.
    Lower = more consistent lengths.
    """
    if not predictions:
        return 0.0
    
    lengths = [len(normalize_answer(p).split()) for p in predictions]
    
    if len(lengths) < 2:
        return 0.0
    
    mean_len = np.mean(lengths)
    std_len = np.std(lengths)
    
    if mean_len == 0:
        return 0.0
    
    return float(std_len / mean_len)

def compute_lexical_diversity(predictions: List[str]) -> float:
    """
    Type-Token Ratio across all predictions.
    Higher diversity = less consistency.
    """
    if not predictions:
        return 0.0
    
    all_tokens = []
    for p in predictions:
        all_tokens.extend(normalize_answer(p).split())
    
    if not all_tokens:
        return 0.0
    
    unique_tokens = len(set(all_tokens))
    total_tokens = len(all_tokens)
    
    return float(unique_tokens / total_tokens)

def compute_entropy(predictions: List[str]) -> float:
    """
    Shannon entropy of normalized predictions.
    Lower entropy = higher consistency (less variation).
    """
    if not predictions:
        return 0.0
    
    normalized = [normalize_answer(p) for p in predictions]
    counts = Counter(normalized)
    total = len(normalized)
    
    probabilities = [count / total for count in counts.values()]
    entropy = -sum(p * np.log2(p) if p > 0 else 0 for p in probabilities)
    
    return float(entropy)

print("Consistency metrics defined")

Accuracy Metrics

In [ ]:
def exact_match(prediction: str, references: List[str]) -> float:
    """Exact match metric"""
    if not references:
        return 0.0
    
    pred = normalize_answer(prediction)
    refs = [normalize_answer(r) for r in references]
    
    return float(any(pred == ref for ref in refs))

def token_f1(prediction: str, references: List[str]) -> float:
    """Token-level F1 score"""
    if not references:
        return 0.0
    
    pred_tokens = normalize_answer(prediction).split()
    ref_token_lists = [normalize_answer(r).split() for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_f1 = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        
        if not common:
            continue
        
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(ref_tokens)
        
        f1 = 2 * precision * recall / (precision + recall)
        max_f1 = max(max_f1, f1)
    
    return float(max_f1)

print("Accuracy metrics defined")

Statistical Utilities

In [ ]:
def bootstrap_ci(data: List[float], n_bootstrap: int = 1000, confidence: float = 0.95) -> Tuple[float, float, float]:
    """Bootstrap confidence intervals"""
    data = np.array(data)
    
    if data.size == 0:
        return 0.0, 0.0, 0.0
    
    if len(data) == 1:
        val = float(data[0])
        return val, val, val
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(sample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    
    return float(np.mean(data)), float(lower), float(upper)

def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Cohen's d effect size"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0.0

def save_json(data: Dict, path: Path):
    """Save JSON with type conversion"""
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.bool_, bool)):
            return bool(obj)
        if isinstance(obj, dict):
            return {key: convert(value) for key, value in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [convert(item) for item in obj]
        return obj
    
    with open(path, 'w') as f:
        json.dump(convert(data), f, indent=2)

print("Statistical utilities defined")

Load Generated Data

In [ ]:
def load_set_c_data(config_name: str) -> Dict:
    """Load Set C data for a configuration"""
    file_path = INPUT_DIR / f"{config_name}_set_c_complete.json"
    
    if not file_path.exists():
        print(f"WARNING: {file_path.name} not found")
        return None
    
    with open(file_path) as f:
        return json.load(f)

configs = [
    'fp16_base',
    'fp16_instruct',
    'awq_base',
    'awq_instruct',
    'nf4_base',
    'nf4_instruct',
    'gptq_base',
    'gptq_instruct'
]

data = {}
for config in configs:
    loaded = load_set_c_data(config)
    if loaded:
        data[config] = loaded
        print(f"Loaded {config}: {len(loaded['samples'])} generations")

print(f"\nTotal configs loaded: {len(data)}")

Organize Data by Sample

In [ ]:
def organize_by_sample(samples: List[Dict]) -> Dict[int, List[Dict]]:
    """Group runs by sample_id"""
    by_sample = defaultdict(list)
    
    for sample in samples:
        sample_id = sample['sample_id']
        by_sample[sample_id].append(sample)
    
    for sample_id in by_sample:
        by_sample[sample_id].sort(key=lambda x: x['run_id'])
    
    return dict(by_sample)

organized_data = {}
for config_name, config_data in data.items():
    organized_data[config_name] = organize_by_sample(config_data['samples'])
    n_samples = len(organized_data[config_name])
    print(f"{config_name}: {n_samples} unique samples")

Compute Sample Consistency Metrics

In [ ]:
def compute_sample_consistency(runs: List[Dict], ground_truth: str, ground_truth_variants: List[str]) -> Dict:
    """Compute consistency metrics for a single sample across runs"""
    
    predictions = [run['rag_prediction'] for run in runs]
    
    extracted = [extract_conservative(p, ground_truth) for p in predictions]
    
    self_bleu = compute_self_bleu(extracted)
    em_consistency = compute_exact_match_consistency(extracted)
    token_overlap = compute_token_overlap_consistency(extracted)
    length_variance = compute_length_variance(extracted)
    lexical_diversity = compute_lexical_diversity(extracted)
    entropy = compute_entropy(extracted)
    
    accuracy_scores = []
    for pred in extracted:
        em = exact_match(pred, ground_truth_variants)
        f1 = token_f1(pred, ground_truth_variants)
        accuracy_scores.append({'em': em, 'f1': f1})
    
    mean_em = np.mean([s['em'] for s in accuracy_scores])
    std_em = np.std([s['em'] for s in accuracy_scores])
    mean_f1 = np.mean([s['f1'] for s in accuracy_scores])
    std_f1 = np.std([s['f1'] for s in accuracy_scores])
    
    return {
        'consistency': {
            'self_bleu': float(self_bleu),
            'exact_match_consistency': float(em_consistency),
            'token_overlap': float(token_overlap),
            'length_variance': float(length_variance),
            'lexical_diversity': float(lexical_diversity),
            'entropy': float(entropy)
        },
        'accuracy': {
            'mean_em': float(mean_em),
            'std_em': float(std_em),
            'mean_f1': float(mean_f1),
            'std_f1': float(std_f1)
        },
        'predictions': extracted
    }

print("Computing consistency metrics for all samples...")

results = {}

for config_name, samples_by_id in organized_data.items():
    print(f"  {config_name}")
    
    config_results = {}
    
    for sample_id, runs in samples_by_id.items():
        ground_truth = runs[0].get('ground_truth', '')
        ground_truth_variants = runs[0].get('ground_truth_variants', [])
        if not ground_truth_variants:
            ground_truth_variants = [ground_truth] if ground_truth else []
        
        category = runs[0].get('category', 'unknown')
        
        sample_consistency = compute_sample_consistency(runs, ground_truth, ground_truth_variants)
        sample_consistency['category'] = category
        sample_consistency['ground_truth'] = ground_truth
        
        config_results[sample_id] = sample_consistency
    
    results[config_name] = config_results

print("Consistency metrics computed")

Aggregate Consistency Metrics

In [ ]:
def aggregate_consistency_metrics(config_results: Dict) -> Dict:
    """Aggregate consistency metrics across all samples"""
    
    consistency_metrics = defaultdict(list)
    accuracy_metrics = defaultdict(list)
    
    for sample_id, sample_data in config_results.items():
        for metric, value in sample_data['consistency'].items():
            consistency_metrics[metric].append(value)
        
        for metric, value in sample_data['accuracy'].items():
            accuracy_metrics[metric].append(value)
    
    aggregated = {
        'consistency': {},
        'accuracy': {}
    }
    
    for metric, values in consistency_metrics.items():
        mean, ci_lower, ci_upper = bootstrap_ci(values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
        
        aggregated['consistency'][metric] = {
            'mean': float(mean),
            'std': float(np.std(values)),
            'median': float(np.median(values)),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper),
            'min': float(np.min(values)),
            'max': float(np.max(values))
        }
    
    for metric, values in accuracy_metrics.items():
        aggregated['accuracy'][metric] = {
            'mean': float(np.mean(values)),
            'std': float(np.std(values))
        }
    
    return aggregated

def aggregate_by_category(config_results: Dict) -> Dict:
    """Aggregate metrics by category"""
    
    by_category = defaultdict(lambda: defaultdict(list))
    
    for sample_id, sample_data in config_results.items():
        category = sample_data['category']
        
        for metric, value in sample_data['consistency'].items():
            by_category[category][metric].append(value)
    
    category_aggregated = {}
    
    for category, metrics in by_category.items():
        category_aggregated[category] = {}
        
        for metric, values in metrics.items():
            category_aggregated[category][metric] = {
                'mean': float(np.mean(values)),
                'std': float(np.std(values)),
                'count': len(values)
            }
    
    return category_aggregated

aggregated_results = {}

for config_name, config_results in results.items():
    aggregated_results[config_name] = {
        'overall': aggregate_consistency_metrics(config_results),
        'by_category': aggregate_by_category(config_results),
        'num_samples': len(config_results)
    }

print("Aggregation complete")

Quantization Consistency Comparison

In [ ]:
def compute_quantization_consistency_comparison(results: Dict, aggregated_results: Dict) -> Dict:
    """Compare consistency between FP16 and quantized models"""
    
    variants = ['base', 'instruct']
    quant_methods = ['awq', 'nf4', 'gptq']
    
    comparison = {}
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in aggregated_results:
            continue
        
        comparison[variant] = {}
        
        fp16_metrics = aggregated_results[fp16_config]['overall']['consistency']
        
        for quant_method in quant_methods:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in aggregated_results:
                continue
            
            quant_metrics = aggregated_results[quant_config]['overall']['consistency']
            
            comparison[variant][quant_method] = {}
            
            for metric in ['self_bleu', 'exact_match_consistency', 'token_overlap', 'entropy']:
                fp16_mean = fp16_metrics[metric]['mean']
                quant_mean = quant_metrics[metric]['mean']
                
                difference = quant_mean - fp16_mean
                relative_diff_pct = (difference / fp16_mean * 100) if fp16_mean != 0 else 0.0
                
                fp16_samples = []
                quant_samples = []
                
                for sample_id in results[fp16_config]:
                    fp16_samples.append(results[fp16_config][sample_id]['consistency'][metric])
                
                for sample_id in results[quant_config]:
                    quant_samples.append(results[quant_config][sample_id]['consistency'][metric])
                
                t_stat, p_value = stats.ttest_ind(fp16_samples, quant_samples)
                effect_size = cohens_d(np.array(fp16_samples), np.array(quant_samples))
                
                comparison[variant][quant_method][metric] = {
                    'fp16_mean': float(fp16_mean),
                    'quant_mean': float(quant_mean),
                    'difference': float(difference),
                    'relative_diff_pct': float(relative_diff_pct),
                    't_statistic': float(t_stat),
                    'p_value': float(p_value),
                    'significant': bool(p_value < ALPHA),
                    'cohens_d': float(effect_size)
                }
    
    return comparison

print("Computing quantization consistency comparison...")
consistency_comparison = compute_quantization_consistency_comparison(results, aggregated_results)
print("Comparison complete")

Accuracy Stability Analysis

In [ ]:
def analyze_accuracy_stability(results: Dict) -> Dict:
    """Analyze how accuracy varies across runs"""
    
    stability_analysis = {}
    
    for config_name, config_results in results.items():
        em_stds = []
        f1_stds = []
        
        for sample_id, sample_data in config_results.items():
            em_stds.append(sample_data['accuracy']['std_em'])
            f1_stds.append(sample_data['accuracy']['std_f1'])
        
        stability_analysis[config_name] = {
            'em_stability': {
                'mean_std': float(np.mean(em_stds)),
                'median_std': float(np.median(em_stds)),
                'max_std': float(np.max(em_stds))
            },
            'f1_stability': {
                'mean_std': float(np.mean(f1_stds)),
                'median_std': float(np.median(f1_stds)),
                'max_std': float(np.max(f1_stds))
            }
        }
    
    return stability_analysis

print("Analyzing accuracy stability...")
accuracy_stability = analyze_accuracy_stability(results)
print("Accuracy stability analysis complete")

Results Summary: Overall Consistency

In [ ]:
print("SET C EVALUATION: OUTPUT CONSISTENCY\n")
print("OVERALL CONSISTENCY METRICS\n")

print(f"{'Config':<20} {'Self-BLEU':<12} {'EM Consist':<12} {'TokOvlp':<12} {'Entropy':<12} {'LexDiv':<12}")

for config_name in configs:
    if config_name not in aggregated_results:
        continue
    
    cons = aggregated_results[config_name]['overall']['consistency']
    
    self_bleu = cons['self_bleu']['mean']
    em_consist = cons['exact_match_consistency']['mean']
    tok_overlap = cons['token_overlap']['mean']
    entropy = cons['entropy']['mean']
    lex_div = cons['lexical_diversity']['mean']
    
    print(f"{config_name:<20} {self_bleu:<12.4f} {em_consist:<12.4f} {tok_overlap:<12.4f} {entropy:<12.4f} {lex_div:<12.4f}")

print("\n\nInterpretation:")
print("  Self-BLEU: Higher = more consistent outputs (less variation)")
print("  EM Consistency: Fraction of output pairs that match exactly")
print("  Token Overlap: Jaccard similarity across runs")
print("  Entropy: Lower = less variation in outputs")
print("  Lexical Diversity: Higher = more varied vocabulary (less consistency)")

Results Summary: Quantization Impact on Consistency

In [ ]:
print("\n\nQUANTIZATION IMPACT ON CONSISTENCY")
print("How quantization affects output stability\n")

for variant in ['base', 'instruct']:
    if variant not in consistency_comparison:
        continue
    
    print(f"\n{variant.upper()} VARIANT:")
    print(f"{'Quant':<10} {'Metric':<20} {'FP16':<12} {'Quant':<12} {'Diff':<12} {'Rel %':<10} {'p-value':<10} {'Sig':<5}")
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in consistency_comparison[variant]:
            continue
        
        for metric in ['self_bleu', 'exact_match_consistency', 'entropy']:
            comp = consistency_comparison[variant][quant_method][metric]
            
            fp16_mean = comp['fp16_mean']
            quant_mean = comp['quant_mean']
            diff = comp['difference']
            rel_diff = comp['relative_diff_pct']
            p_val = comp['p_value']
            
            sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
            
            print(f"{quant_method:<10} {metric:<20} {fp16_mean:<12.4f} {quant_mean:<12.4f} {diff:<+12.4f} {rel_diff:<+10.2f} {p_val:<10.4f} {sig:<5}")

print("\n\nInterpretation:")
print("  Positive difference in Self-BLEU/EM Consistency: Quantized model MORE consistent")
print("  Negative difference in Entropy: Quantized model MORE consistent")
print("  Consistency changes can indicate loss of model expressiveness")

Results Summary: Accuracy Stability

In [ ]:
print("\n\nACCURACY STABILITY ACROSS RUNS")
print("How much accuracy varies with same prompt\n")

print(f"{'Config':<20} {'Mean EM Std':<15} {'Median EM Std':<15} {'Mean F1 Std':<15} {'Median F1 Std':<15}")

for config_name in configs:
    if config_name not in accuracy_stability:
        continue
    
    stab = accuracy_stability[config_name]
    
    mean_em_std = stab['em_stability']['mean_std']
    median_em_std = stab['em_stability']['median_std']
    mean_f1_std = stab['f1_stability']['mean_std']
    median_f1_std = stab['f1_stability']['median_std']
    
    print(f"{config_name:<20} {mean_em_std:<15.4f} {median_em_std:<15.4f} {mean_f1_std:<15.4f} {median_f1_std:<15.4f}")

print("\n\nInterpretation:")
print("  Lower standard deviation = more stable accuracy across runs")
print("  High variance suggests unpredictable behavior with same input")

Results Summary: Category Breakdown

In [ ]:
print("\n\nCONSISTENCY BY CATEGORY\n")

all_categories = set()
for config_name in aggregated_results:
    all_categories.update(aggregated_results[config_name]['by_category'].keys())

for category in sorted(all_categories):
    print(f"\n{category.upper()}:")
    print(f"{'Config':<20} {'Self-BLEU':<12} {'EM Consist':<12} {'Entropy':<12} {'N':<8}")
    
    for config_name in configs:
        if config_name not in aggregated_results:
            continue
        
        if category not in aggregated_results[config_name]['by_category']:
            continue
        
        cat_data = aggregated_results[config_name]['by_category'][category]
        
        self_bleu = cat_data['self_bleu']['mean']
        em_consist = cat_data['exact_match_consistency']['mean']
        entropy = cat_data['entropy']['mean']
        count = cat_data['self_bleu']['count']
        
        print(f"{config_name:<20} {self_bleu:<12.4f} {em_consist:<12.4f} {entropy:<12.4f} {count:<8}")

Cross-Model Consistency Ranking

In [ ]:
print("\n\nCONSISTENCY RANKINGS")
print("Models ranked by output consistency (Self-BLEU)\n")

consistency_scores = []

for config_name in configs:
    if config_name not in aggregated_results:
        continue
    
    self_bleu = aggregated_results[config_name]['overall']['consistency']['self_bleu']['mean']
    consistency_scores.append((config_name, self_bleu))

consistency_scores.sort(key=lambda x: x[1], reverse=True)

print(f"{'Rank':<6} {'Config':<20} {'Self-BLEU':<12}")
for i, (config_name, score) in enumerate(consistency_scores, 1):
    print(f"{i:<6} {config_name:<20} {score:<12.4f}")

print("\n\nInterpretation:")
print("  Higher-ranked models produce more consistent outputs")
print("  This can be positive (reliability) or negative (lack of expressiveness)")

Consistency-Accuracy Tradeoff

In [ ]:
print("\n\nCONSISTENCY-ACCURACY TRADEOFF\n")

print(f"{'Config':<20} {'Self-BLEU':<12} {'Mean F1':<12} {'F1 Std':<12}")

for config_name in configs:
    if config_name not in aggregated_results:
        continue
    
    self_bleu = aggregated_results[config_name]['overall']['consistency']['self_bleu']['mean']
    mean_f1 = aggregated_results[config_name]['overall']['accuracy']['mean_f1']['mean']
    f1_std = accuracy_stability[config_name]['f1_stability']['mean_std']
    
    print(f"{config_name:<20} {self_bleu:<12.4f} {mean_f1:<12.4f} {f1_std:<12.4f}")

print("\n\nInterpretation:")
print("  Ideal: High consistency (Self-BLEU) + High accuracy + Low F1 Std")
print("  Models can be consistent but wrong, or inconsistent but sometimes right")

Statistical Tests

In [ ]:
print("\n\nSTATISTICAL TESTS")
print("Testing if consistency differs significantly across quantization\n")

for variant in ['base', 'instruct']:
    fp16_config = f'fp16_{variant}'
    
    if fp16_config not in results:
        continue
    
    print(f"\n{variant.upper()} VARIANT (Self-BLEU):")
    print(f"{'Comparison':<30} {'t-stat':<12} {'p-value':<12} {'Cohen d':<12} {'Sig':<5}")
    
    fp16_samples = [results[fp16_config][sid]['consistency']['self_bleu'] for sid in results[fp16_config]]
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        quant_config = f'{quant_method}_{variant}'
        
        if quant_config not in results:
            continue
        
        quant_samples = [results[quant_config][sid]['consistency']['self_bleu'] for sid in results[quant_config]]
        
        t_stat, p_value = stats.ttest_ind(fp16_samples, quant_samples)
        effect_size = cohens_d(np.array(fp16_samples), np.array(quant_samples))
        
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        
        comparison_name = f"FP16 vs {quant_method.upper()}"
        print(f"{comparison_name:<30} {t_stat:<+12.4f} {p_value:<12.6f} {effect_size:<+12.4f} {sig:<5}")

Save Complete Results

In [ ]:
final_results = {
    'evaluation': 'Set C - Output Consistency',
    'configs': {
        config_name: {
            'overall': config_data['overall'],
            'by_category': config_data['by_category'],
            'num_samples': config_data['num_samples']
        }
        for config_name, config_data in aggregated_results.items()
    },
    'quantization_comparison': consistency_comparison,
    'accuracy_stability': accuracy_stability,
    'metadata': {
        'n_bootstrap': N_BOOTSTRAP,
        'confidence_level': CONFIDENCE_LEVEL,
        'alpha': ALPHA,
        'random_seed': RANDOM_SEED,
        'consistency_runs': CONSISTENCY_RUNS,
        'metrics': {
            'consistency': ['self_bleu', 'exact_match_consistency', 'token_overlap', 'length_variance', 'lexical_diversity', 'entropy'],
            'accuracy': ['exact_match', 'token_f1']
        },
        'analyses': ['overall_consistency', 'category_consistency', 'quantization_impact', 'accuracy_stability']
    }
}

output_path = OUTPUT_DIR / 'set_c_complete_results.json'
save_json(final_results, output_path)
print(f"\nComplete results saved to: {output_path}")

consistency_summary = []

for config_name in configs:
    if config_name not in aggregated_results:
        continue
    
    cons = aggregated_results[config_name]['overall']['consistency']
    acc = aggregated_results[config_name]['overall']['accuracy']
    stab = accuracy_stability[config_name]
    
    row = {
        'config': config_name,
        'self_bleu': cons['self_bleu']['mean'],
        'em_consistency': cons['exact_match_consistency']['mean'],
        'token_overlap': cons['token_overlap']['mean'],
        'entropy': cons['entropy']['mean'],
        'lexical_diversity': cons['lexical_diversity']['mean'],
        'mean_f1': acc['mean_f1']['mean'],
        'f1_stability': stab['f1_stability']['mean_std']
    }
    
    consistency_summary.append(row)

consistency_df = pd.DataFrame(consistency_summary)
csv_path = OUTPUT_DIR / 'consistency_summary.csv'
consistency_df.to_csv(csv_path, index=False)
print(f"Consistency summary CSV: {csv_path}")

category_breakdown = []

for config_name in configs:
    if config_name not in aggregated_results:
        continue
    
    for category, cat_data in aggregated_results[config_name]['by_category'].items():
        row = {
            'config': config_name,
            'category': category,
            'self_bleu': cat_data['self_bleu']['mean'],
            'em_consistency': cat_data['exact_match_consistency']['mean'],
            'entropy': cat_data['entropy']['mean'],
            'count': cat_data['self_bleu']['count']
        }
        
        category_breakdown.append(row)

category_df = pd.DataFrame(category_breakdown)
csv_path = OUTPUT_DIR / 'by_category_consistency.csv'
category_df.to_csv(csv_path, index=False)
print(f"Category breakdown CSV: {csv_path}")

quant_comparison_summary = []

for variant in ['base', 'instruct']:
    if variant not in consistency_comparison:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in consistency_comparison[variant]:
            continue
        
        row = {
            'variant': variant,
            'quant_method': quant_method
        }
        
        for metric in ['self_bleu', 'exact_match_consistency', 'entropy']:
            comp = consistency_comparison[variant][quant_method][metric]
            row[f'{metric}_difference'] = comp['difference']
            row[f'{metric}_relative_pct'] = comp['relative_diff_pct']
            row[f'{metric}_p_value'] = comp['p_value']
        
        quant_comparison_summary.append(row)

quant_comparison_df = pd.DataFrame(quant_comparison_summary)
csv_path = OUTPUT_DIR / 'quantization_consistency_comparison.csv'
quant_comparison_df.to_csv(csv_path, index=False)
print(f"Quantization comparison CSV: {csv_path}")

print("\nEVALUATION COMPLETE")
print("Set C includes: Output Consistency, Accuracy Stability, and Quantization Impact on Determinism")